# Parquet Basics — 03: Partitioning

## What you will learn
Partitioning is the most impactful optimization for Parquet at scale.
A well-partitioned table makes queries 10-100x faster by eliminating
unnecessary I/O at the directory level — before even opening a file.

In this notebook:
1. What partitioning is and how it works physically
2. `partitionBy` — choosing the right column
3. Partition pruning — how Spark skips entire directories
4. Cardinality — the most common partitioning mistake
5. Multi-level partitioning — date/region hierarchies
6. Dynamic partition overwrite — replacing only changed partitions


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

## Step 1 — What Partitioning Does Physically

Without partitioning:
```
output/
  part-00000.parquet   ← all data mixed together
  part-00001.parquet
  part-00002.parquet
```

With `partitionBy("category")`:
```
output/
  category=Books/
    part-00000.parquet   ← only Books rows
  category=Electronics/
    part-00000.parquet   ← only Electronics rows
  category=Clothing/
    part-00000.parquet   ← only Clothing rows
```

When you query `WHERE category = 'Electronics'`:
- Without partitioning → Spark opens ALL files, filters in memory
- With partitioning → Spark opens ONLY `category=Electronics/` directory


In [ ]:
import os

# Write without partitioning
no_part = f"{DATA_DIR}/no_partition"
df.write.mode("overwrite").parquet(no_part)

# Write with partitioning by category
with_part = f"{DATA_DIR}/with_partition"
df.write.mode("overwrite").partitionBy("category").parquet(with_part)

# Show directory structure
print("Without partitioning:")
files = glib.glob(f"{no_part}/*.parquet")
print(f"  {len(files)} flat parquet files")
print()

print("With partitionBy('category'):")
for root, dirs, files in os.walk(with_part):
    level = root.replace(with_part, '').count(os.sep)
    if level <= 1:
        indent = '  ' * level
        folder = os.path.basename(root)
        n_files = len([f for f in files if f.endswith('.parquet')])
        if folder:
            rows = spark.read.parquet(root).count() if n_files > 0 else 0
            size_kb = sum(pathlib.Path(root+"/"+f).stat().st_size
                         for f in files if f.endswith('.parquet')) // 1024
            print(f"  {indent}{folder}/ → {n_files} file(s), {rows:,} rows, {size_kb} KB")

## Step 2 — Partition Pruning: The Key Benefit


In [ ]:
runs = 3

def timed(label, fn):
    times = []
    for _ in range(runs):
        t0 = time.time(); fn(); times.append(time.time()-t0)
    return sorted(times)[1]

# Query: total revenue for Electronics only
query_no_part = lambda: (spark.read.parquet(no_part)
    .filter(F.col("category") == "Electronics")
    .agg(F.sum("revenue")).collect())

query_with_part = lambda: (spark.read.parquet(with_part)
    .filter(F.col("category") == "Electronics")
    .agg(F.sum("revenue")).collect())

t_no   = timed("No partition",   query_no_part)
t_with = timed("With partition", query_with_part)

print("Partition pruning benchmark (WHERE category = Electronics):")
print(f"  No partition   : {t_no:.3f}s  (reads ALL {len(glib.glob(no_part+'/*.parquet'))} files)")
print(f"  With partition : {t_with:.3f}s  (reads ONLY category=Electronics/ directory)")
print(f"  Speedup        : {t_no/t_with:.1f}x")
print()

# Verify partition pruning in explain
print("Explain plan — look for PartitionFilters:")
spark.read.parquet(with_part) \
     .filter(F.col("category") == "Electronics") \
     .explain(mode="formatted")

## Step 3 — Cardinality: The Most Common Partitioning Mistake

Partition column cardinality = number of distinct values.
- **Too low** (< 5 values): no benefit, huge files per partition
- **Sweet spot** (10–1000 values): good balance
- **Too high** (> 10000 values): too many tiny files, massive metadata overhead

**Never partition by**: user_id, order_id, timestamp, UUID, email
**Good to partition by**: year, month, date, country, category, status


In [ ]:
# Demonstrate cardinality impact
print("Partition cardinality comparison:")
print()

# Low cardinality: 6 categories (good)
low_card = f"{DATA_DIR}/low_cardinality"
df.write.mode("overwrite").partitionBy("category").parquet(low_card)
low_dirs = [d for d in os.listdir(low_card) if not d.startswith("_")]
print(f"Low cardinality (category — {len(low_dirs)} values):")
print(f"  Partition dirs : {len(low_dirs)}")
for d in sorted(low_dirs):
    size_kb = sum(pathlib.Path(f"{low_card}/{d}/{f}").stat().st_size
                  for f in os.listdir(f"{low_card}/{d}") if f.endswith(".parquet")) // 1024
    print(f"    {d}: {size_kb} KB")

print()

# High cardinality: customer_id (bad — 50K distinct values!)
high_card = f"{DATA_DIR}/high_cardinality"
# Only use 100 customers to avoid creating 50K directories
df.withColumn("customer_bucket",
              (F.col("customer_id") % 100).cast("string")) \
  .write.mode("overwrite").partitionBy("customer_bucket").parquet(high_card)
high_dirs = [d for d in os.listdir(high_card) if not d.startswith("_")]
print(f"Medium cardinality (customer_id % 100 — {len(high_dirs)} values):")
print(f"  Partition dirs : {len(high_dirs)}")
avg_kb = sum(
    sum(pathlib.Path(f"{high_card}/{d}/{f}").stat().st_size
        for f in os.listdir(f"{high_card}/{d}") if f.endswith(".parquet"))
    for d in high_dirs
) / len(high_dirs) / 1024
print(f"  Avg partition size: {avg_kb:.1f} KB (too small — small file problem!)")

## Step 4 — Multi-Level Partitioning: Date Hierarchies


In [ ]:
# Common production pattern: partition by year/month
multi_path = f"{DATA_DIR}/multi_level"

df.withColumn("year",  F.year("order_date")) \
  .withColumn("month", F.month("order_date")) \
  .write.mode("overwrite") \
  .partitionBy("category", "year", "month") \
  .parquet(multi_path)

print("Multi-level partitioning (category / year / month):")
for root, dirs, files in os.walk(multi_path):
    level = root.replace(multi_path, '').count(os.sep)
    if level <= 3 and level > 0:
        n_pq = len([f for f in files if f.endswith('.parquet')])
        if n_pq > 0:
            print("  " * level + os.path.basename(root) + f"/ [{n_pq} file(s)]")

print()
# Partition pruning on multiple levels
t0 = time.time()
result = spark.read.parquet(multi_path) \
              .filter(
                  (F.col("category") == "Electronics") &
                  (F.col("year") == 2023) &
                  (F.col("month") == 6)
              ).agg(F.sum("revenue"), F.count("*")).collect()
t_multi = time.time() - t0
print(f"Multi-level filter (Electronics / 2023 / June): {t_multi:.3f}s")
print(f"Result: {result[0]}")

## Step 5 — Dynamic Partition Overwrite

Default `overwrite` replaces the ENTIRE table.
Dynamic overwrite replaces ONLY the partitions present in the new data.


In [ ]:
# Initial full table
full_path = f"{DATA_DIR}/dynamic_overwrite"
df.withColumn("month", F.month("order_date")) \
  .write.mode("overwrite").partitionBy("category","month").parquet(full_path)

initial_count = spark.read.parquet(full_path).count()
print(f"Initial table: {initial_count:,} rows")

# Simulate: reprocess only Electronics for January
jan_elec = df.filter(
    (F.col("category") == "Electronics") &
    (F.month("order_date") == 1)
).withColumn("month", F.lit(1))

print(f"Reprocessing Electronics/January: {jan_elec.count():,} rows")

# Static overwrite — replaces EVERYTHING
jan_elec.write.mode("overwrite").partitionBy("category","month").parquet(full_path)
static_count = spark.read.parquet(full_path).count()
print(f"After STATIC overwrite: {static_count:,} rows  ← WRONG: lost all other data!")

# Restore
df.withColumn("month", F.month("order_date")) \
  .write.mode("overwrite").partitionBy("category","month").parquet(full_path)

# Dynamic overwrite — replaces ONLY Electronics/January
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
jan_elec.write.mode("overwrite").partitionBy("category","month").parquet(full_path)
dynamic_count = spark.read.parquet(full_path).count()
print(f"After DYNAMIC overwrite: {dynamic_count:,} rows  ← CORRECT: only Jan/Electronics replaced")

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

## Summary: Partitioning Best Practices

### Choosing the right partition column
| Column type | Cardinality | Recommendation |
|---|---|---|
| `order_date` (daily) | ~365/year | ✅ Excellent — time-based pruning |
| `category` | 5–20 | ✅ Good — low cardinality |
| `country` / `region` | 10–200 | ✅ Good |
| `year` + `month` | 12/year | ✅ Classic pattern |
| `user_id` | Millions | ❌ Too high — small file problem |
| `order_id` | Unique | ❌ Never partition by unique keys |

### Partition size target
- Aim for **128 MB – 1 GB per partition**
- Too small (< 10 MB) → small file problem → slow metadata listing
- Too large (> 5 GB) → no skipping benefit → slow scans

### Commands
```python
# Standard partition
df.write.partitionBy("category").parquet(path)

# Multi-level
df.write.partitionBy("year", "month", "category").parquet(path)

# Dynamic overwrite (replace only affected partitions)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
df.write.mode("overwrite").partitionBy("category").parquet(path)
```
